In [ ]:
import pandas as pd
import os
import re
import ssl
import tkinter as tk
from tkinter import filedialog

# --- CHECK FOR NLTK LIBRARY ---
try:
    import nltk
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
except ImportError:
    print("NLTK library missing. Installing...")
    !pip install nltk
    import nltk
    from nltk.sentiment.vader import SentimentIntensityAnalyzer

# --- FIX SSL ERRORS (Common on Mac/Corporate Wifi) ---
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download VADER lexicon
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    print("Downloading sentiment dictionary...")
    nltk.download('vader_lexicon', quiet=True)

# Initialize
sia = SentimentIntensityAnalyzer()
print("Setup complete. VADER initialized.")

In [ ]:
# --- 1. CUSTOM IDIOMS (Fixing VADER's blind spots) ---
# Scale: -4.0 (Extreme Negative) to +4.0 (Extreme Positive)
CUSTOM_IDIOMS = {
    "cost an arm and a leg": -3.5,
    "break the bank": -3.0,
    "rip off": -3.5,
    "cherry on top": 3.0,
    "piece of cake": 2.0,
    "not my cup of tea": -2.0,
    "meh": -1.5,
    "game changer": 3.5,
    "hidden gem": 3.5,
    "middle of nowhere": -2.0,
    "below average": -2.0,
    "nothing to write home about": -1.0,
    "blew me away": 3.5,
    "steer clear": -3.0
}

# Update VADER with your custom rules
sia.lexicon.update(CUSTOM_IDIOMS)
print("Custom idioms added to dictionary.")

# --- 2. DEFINE TOPICS ---
TOPIC_KEYWORDS = {
    "Staff_Attitude": [
        "staff", "attitude", "reception", "host", "service", "friendly", 
        "rude", "helpful", "manager", "guard", "check-in", "check in",
        "welcoming", "polite", "unpleasant", "support", "personnel"
    ],
    "Cleanliness": [
        "clean", "dirty", "smell", "dust", "mold", "stain", "hygiene", 
        "messy", "tidy", "bug", "insect", "cockroach", "rat", "hair",
        "filthy", "spotted", "trash", "garbage"
    ],
    "Room_Comfort": [
        "room", "bed", "bathroom", "shower", "ac", "air con", "conditioner",
        "view", "window", "sleep", "noise", "pillow", "comfortable", 
        "spacious", "small", "tiny", "huge", "soft", "hard", "furniture"
    ],
    "Location": [
        "location", "center", "far", "close", "near", "district", "walk", 
        "airport", "grab", "taxi", "convenient", "traffic", "accessible"
    ],
    "Price": [
        "price", "value", "expensive", "cheap", "worth", "money", "cost",
        "budget", "deal", "affordable"
    ],
    "Facilities": [
        "pool", "gym", "wifi", "internet", "elevator", "lift", "breakfast", "food",
        "parking", "lobby", "amenities"
    ]
}

In [ ]:
def get_sentiment_score(text_segment):
    """
    Returns a score from -10 to 10 based on VADER compound score.
    """
    compound_score = sia.polarity_scores(text_segment)['compound']
    # Scale -1.0...1.0 to -10...10
    return round(compound_score * 10)

def analyze_review_aspects(text):
    """
    Splits review into segments and scores topics found in each segment.
    """
    if not isinstance(text, str): return {}
    
    text = text.lower()
    scores = {topic: [] for topic in TOPIC_KEYWORDS}
    
    # Split text by punctuation or contrasting conjunctions
    segments = re.split(r'[.!?,;]|\bbut\b|\bhowever\b|\balthough\b', text)
    
    for segment in segments:
        if len(segment.strip()) < 2: continue
        
        segment_score = get_sentiment_score(segment)
        
        for topic, keywords in TOPIC_KEYWORDS.items():
            if any(word in segment for word in keywords):
                scores[topic].append(segment_score)
    
    # Average the scores
    final_scores = {}
    for topic, val_list in scores.items():
        if val_list:
            avg = sum(val_list) / len(val_list)
            final_scores[f"Score_{topic}"] = round(avg)
        else:
            final_scores[f"Score_{topic}"] = None
            
    return final_scores


In [ ]:
# --- FILE SELECTION ---
# Tries to find 'raw.csv' automatically, otherwise opens popup
input_file = "raw.csv"

if not os.path.exists(input_file):
    print("Could not find 'raw.csv'. Opening file picker...")
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        input_file = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
        root.destroy()
    except:
        print("File picker failed. Please set 'input_file' manually.")

if input_file:
    print(f"Loading: {input_file}")
    # utf-8-sig handles Excel files better
    df = pd.read_csv(input_file, encoding='utf-8-sig', on_bad_lines='skip')
    
    # Standardize column name
    if 'Comment' not in df.columns and 'Review' in df.columns:
        df.rename(columns={'Review': 'Comment'}, inplace=True)
    
    print(f"Loaded {len(df)} rows.")
    display(df.head(3)) # Jupyter command to show table
else:
    print("No file selected.")

In [ ]:
print("Analyzing sentiments...")

# Apply logic
score_data = df['Comment'].apply(analyze_review_aspects).apply(pd.Series)
result_df = pd.concat([df, score_data], axis=1)

print("Analysis Complete.")
display(result_df.head())

In [ ]:
output_file = "agoda_data_scored.csv"

try:
    result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"Success! Saved to {output_file}")
except PermissionError:
    print(f"ERROR: Please close {output_file} in Excel and run this cell again.")